# SalesInsight-PY

## Criando o Dataset de vendas

In [1]:
import pandas as pd
import numpy as np
import random
from datetime import datetime, timedelta
 # Função abaixo foi usado para gerar um dataset randomico.
def gerar_dataset_vendas(n_registros=2000, seed=42):
    """Gera um dataset sintetico de vendas com dados sujos."""
    random.seed(seed)
    np.random.seed(seed)

    produtos = ["Notebook", "Smartphone", "Tablet", "Monitor",
                "Teclado", "Mouse", "Headset"]
    categorias = {"Notebook": "Computadores", "Smartphone": "Celulares",
                  "Tablet": "Celulares", "Monitor": "Computadores",
                  "Teclado": "Perifericos", "Mouse": "Perifericos",
                  "Headset": "Perifericos"}
    precos = {"Notebook": 3500, "Smartphone": 2200, "Tablet": 1800,
              "Monitor": 1200, "Teclado": 250, "Mouse": 120,
              "Headset": 350}
    regioes = ["Sudeste", "Sul", "Nordeste", "Centro-Oeste", "Norte"]

    data_inicio = datetime(2025, 1, 1)
    dados = []

    for i in range(n_registros):
        produto = random.choice(produtos)
        categoria = categorias[produto]
        quantidade = random.randint(1, 10)
        preco = round(precos[produto] * random.uniform(0.85, 1.15), 2)
        data = data_inicio + timedelta(days=random.randint(0, 364))
        data_txt = data.strftime("%Y-%m-%d")
        cliente = f"Cliente_{random.randint(1, 50):03d}"

        # --- sujeira proposital para a etapa de limpeza --- 
        if random.random() < 0.03:
        	data_txt = "DATA INVALIDA"       	# data invalida

        dados.append({
            "id_venda": i + 1,
            "data_venda": data_txt,
            "cliente": cliente,
            "produto": produto,
            "categoria": categoria,
            "regiao": random.choice(regioes),
            "quantidade": quantidade,
            "preco_unitario": preco,
        })

    return pd.DataFrame(dados)


# Gerar e salvar o CSV bruto
df_bruto = gerar_dataset_vendas()
df_bruto.to_csv("vendas.csv", index=False)
print(f"Dataset gerado com {len(df_bruto)} registros.")
print(df_bruto.head())

Dataset gerado com 2000 registros.
   id_venda     data_venda      cliente     produto     categoria    regiao  \
0         1     2025-05-21  Cliente_016       Mouse   Perifericos   Sudeste   
1         2  DATA INVALIDA  Cliente_003       Mouse   Perifericos       Sul   
2         3     2025-10-15  Cliente_013  Smartphone     Celulares     Norte   
3         4     2025-05-23  Cliente_001     Monitor  Computadores       Sul   
4         5     2025-03-21  Cliente_014       Mouse   Perifericos  Nordeste   

   quantidade  preco_unitario  
0           2          102.90  
1           9          105.13  
2           9         2267.33  
3           4         1181.72  
4           7          114.25  


## Inspecionar e Descrever os Dados

In [2]:
dados = pd.read_csv("vendas.csv") # ler o dataset gerado randomicamente
def inspecionar_dados(df):
    print("=== INSPECAO INICIAL DO DATASET ===")
    print("Shape:", df.shape) # verificar tamanho do indice e coluna do dataset
    print("Colunas:\n", list(df.columns)) # listar as colunas no dataset
    print("Tipos de dados:\n", df.dtypes) #quais os tipos de dados de cada coluna
    print("Valores nulos por coluna:\n", df.isnull().sum()) #verificação de valores nulos no dataset que no caso não ha nenhum registro vazio
    print("Primeiros registros:\n", df.head)
    return df

inspec_inicial = inspecionar_dados(dados) # chamando a função inspecionar dados da variavel dados ('vendas.csv') e declarando ela como inspeção inicial

=== INSPECAO INICIAL DO DATASET ===
Shape: (2000, 8)
Colunas:
 ['id_venda', 'data_venda', 'cliente', 'produto', 'categoria', 'regiao', 'quantidade', 'preco_unitario']
Tipos de dados:
 id_venda            int64
data_venda            str
cliente               str
produto               str
categoria             str
regiao                str
quantidade          int64
preco_unitario    float64
dtype: object
Valores nulos por coluna:
 id_venda          0
data_venda        0
cliente           0
produto           0
categoria         0
regiao            0
quantidade        0
preco_unitario    0
dtype: int64
Primeiros registros:
 <bound method NDFrame.head of       id_venda     data_venda      cliente     produto     categoria  \
0            1     2025-05-21  Cliente_016       Mouse   Perifericos   
1            2  DATA INVALIDA  Cliente_003       Mouse   Perifericos   
2            3     2025-10-15  Cliente_013  Smartphone     Celulares   
3            4     2025-05-23  Cliente_001     Monitor

## Tratar os Dados (datetime)

In [3]:
def tratar_dados(df):
     """  Trata o DataFrame de vendas.  Retorna: (df_limpo, relatorio), onde relatorio e um dicionario  
               com as contagens de registros iniciais, excluidos e finais.  """
     df = df.copy() # copiando o dataset para não alterar o arquivo original
     # Remover espacos extras nas colunas de texto
     df.columns = df.columns.str.strip()

     registros_iniciais = len(df) # atribuindo o tamanho do indice do dataset
     
     # Converter data_venda e descartar datas invalidas
     df["data_venda"] = pd.to_datetime(df["data_venda"], errors="coerce")
     registros_excluidos = df["data_venda"].isna().sum() # contabilizar os erros com .isna() e .sum() para contar os erros
     df_limpos = df.dropna(subset=["data_venda"]).copy() # entregando para a nova variavel df_limpos o dataset após excluir com o comando .dropna os erros encontrados
     
     # Ajustar os tipos numericos
     df_limpos["quantidade"] = df_limpos["quantidade"].astype(int)
     df_limpos["preco_unitario"] = df_limpos["preco_unitario"].astype(float)
     # print(df[["quantidade", "preco_unitario"]].dtypes)
     # Exibir os tipos numericos de quantidade e preco_unitario.
     
     # Montar e imprimir o relatorio de limpeza
     relatorio = {
          "registros_iniciais": registros_iniciais,
          "registros_excluidos": int(registros_excluidos),
          "registros_finais": len(df_limpos),
     }
     print(relatorio)

     return df_limpos, relatorio

dados_limpos, relatorio = tratar_dados(dados)

{'registros_iniciais': 2000, 'registros_excluidos': 65, 'registros_finais': 1935}


## Criar Colunas Derivadas com Transformações Condicionais

In [4]:
def criar_colunas_derivadas(df):
    """Cria as colunas derivadas a partir do DataFrame já limpo."""
    df = df.copy()

    # Receita total: quantidade x preço unitário
    df["receita_total"] = df["quantidade"] * df["preco_unitario"]

    # Mês numérico e nome do mês em português
    df["mes"] = df["data_venda"].dt.month
    meses_pt = {
        1: "Janeiro", 2: "Fevereiro", 3: "Março", 4: "Abril",
        5: "Maio", 6: "Junho", 7: "Julho", 8: "Agosto",
        9: "Setembro", 10: "Outubro", 11: "Novembro", 12: "Dezembro"
    }
    df["mes_nome"] = df["mes"].map(meses_pt)

    # Trimestre e ano
    df["trimestre"] = "Q" + df["data_venda"].dt.quarter.astype(str)
    df["ano"] = df["data_venda"].dt.year

    # Classificação condicional da receita por item
    df["faixa_receita_item"] = "Baixa"
    df.loc[df["receita_total"] >= 500, "faixa_receita_item"] = "Média"
    df.loc[df["receita_total"] >= 1000, "faixa_receita_item"] = "Alta"
    return df


# Aplicar as transformações somente após a limpeza das datas
dados_limpos = criar_colunas_derivadas(dados_limpos)

print("Colunas derivadas criadas com sucesso!")
print(dados_limpos[[
    "data_venda", "quantidade", "preco_unitario", "receita_total",
    "mes", "mes_nome", "trimestre", "ano", "faixa_receita_item"
]].head())

print("\nDistribuição por faixa de receita:")
print(dados_limpos["faixa_receita_item"].value_counts())

Colunas derivadas criadas com sucesso!
  data_venda  quantidade  preco_unitario  receita_total  mes mes_nome  \
0 2025-05-21           2          102.90         205.80    5     Maio   
2 2025-10-15           9         2267.33       20405.97   10  Outubro   
3 2025-05-23           4         1181.72        4726.88    5     Maio   
4 2025-03-21           7          114.25         799.75    3    Março   
5 2025-07-03           2         3373.92        6747.84    7    Julho   

  trimestre   ano faixa_receita_item  
0        Q2  2025              Baixa  
2        Q4  2025               Alta  
3        Q2  2025               Alta  
4        Q1  2025              Média  
5        Q3  2025               Alta  

Distribuição por faixa de receita:
faixa_receita_item
Alta     1532
Média     213
Baixa     190
Name: count, dtype: int64


## Calcular Métricas Agregadas com groupby

In [5]:
def calcular_metricas(df):
    """
    Calcula métricas agregadas do dataset.

    Retorna um dicionário de DataFrames com as chaves:
    por_mes, top_produtos, por_categoria e por_regiao.
    """

    metricas = {}

    # Receita total, quantidade vendida e número de vendas por mês
    metricas["por_mes"] = (df.groupby("mes").agg(receita_total=("receita_total", "sum"),
    quantidade=("quantidade", "sum"), n_vendas=("data_venda", "count")).sort_index().reset_index())

    # Top 5 produtos por receita total
    metricas["top_produtos"] = (df.groupby("produto").agg(receita_total=("receita_total", "sum"))
    .sort_values(by="receita_total",ascending=False).head(5).reset_index())

    # Receita total por categoria
    metricas["por_categoria"] = (df.groupby("categoria").agg(receita_total=("receita_total", "sum"))
    .sort_values(by="receita_total",ascending=False).reset_index())

    # Receita total e ticket médio por região
    metricas["por_regiao"] = (df.groupby("regiao").agg(receita_total=("receita_total", "sum"),
    ticket_medio=("receita_total", "mean")).sort_values(by="receita_total",ascending=False).reset_index())

    return metricas


# Calcular as métricas usando o dataset limpo
metricas = calcular_metricas(dados_limpos)

# Exibir os resultados no console
print("\n=== POR MÊS ===")
print(metricas["por_mes"].to_string(index=False))

print("\n=== TOP 5 PRODUTOS ===")
print(metricas["top_produtos"].to_string(index=False))

print("\n=== POR CATEGORIA ===")
print(metricas["por_categoria"].to_string(index=False))

print("\n=== POR REGIÃO ===")
print(metricas["por_regiao"].to_string(index=False))


=== POR MÊS ===
 mes  receita_total  quantidade  n_vendas
   1     1021509.43         809       156
   2     1245781.41         937       160
   3     1216221.11         867       157
   4     1076421.32         881       157
   5     1021307.32         773       140
   6     1006752.08         826       157
   7     1210764.88         827       158
   8     1241209.29         932       165
   9     1172255.68         893       168
  10     1337802.29        1042       186
  11     1124914.91         869       154
  12     1380269.87        1019       177

=== TOP 5 PRODUTOS ===
   produto  receita_total
  Notebook     4905935.08
Smartphone     3350290.74
    Tablet     2740096.52
   Monitor     1983396.40
   Headset      510623.14

=== POR CATEGORIA ===
   categoria  receita_total
Computadores     6889331.48
   Celulares     6090387.26
 Perifericos     1075490.85

=== POR REGIÃO ===
      regiao  receita_total  ticket_medio
         Sul     2917259.08   7738.087745
     Sudeste     2

## Segmentar Clientes por Nível de Gasto

In [6]:
def segmentar_clientes(df):
    """
    Agrupa por cliente, calcula o total gasto e classifica
    em Bronze, Prata ou Ouro.

    Retorna um DataFrame com:
    cliente, total_gasto, segmento.
    """

    # Agrupar clientes e calcular o gasto acumulado
    clientes = (df.groupby("cliente").agg(total_gasto=("receita_total", "sum")).reset_index())

    # Classificar clientes com lambda e apply
    clientes["segmento"] = clientes["total_gasto"].apply(lambda gasto:("Bronze" if gasto < 200000 else "Prata" if gasto <= 400000 else "Ouro"))

    # Ordenar do maior para o menor gasto
    clientes = clientes.sort_values(by="total_gasto", ascending=False).reset_index(drop=True)

    return clientes


# Executar a segmentação usando o dataset limpo
clientes_segmentados = segmentar_clientes(dados_limpos)

# Exibir os 10 maiores clientes
print("\n=== TOP 10 CLIENTES POR GASTO ===")
print(clientes_segmentados.head(10).to_string(index=False))

# Exibir a distribuição de clientes por segmento
print("\n=== DISTRIBUIÇÃO POR SEGMENTO ===")
print(clientes_segmentados["segmento"].value_counts().reindex(["Bronze", "Prata", "Ouro"], fill_value=0))

# Exibir a distribuição percentual
print("\n=== DISTRIBUIÇÃO PERCENTUAL ===")
print((clientes_segmentados["segmento"].value_counts(normalize=True).reindex(["Bronze", "Prata", "Ouro"], fill_value=0).mul(100).round(2)).astype(str) + "%")



=== TOP 10 CLIENTES POR GASTO ===
    cliente  total_gasto segmento
Cliente_043    490172.44     Ouro
Cliente_008    481333.70     Ouro
Cliente_049    427969.67     Ouro
Cliente_048    426967.67     Ouro
Cliente_037    423937.36     Ouro
Cliente_045    420368.06     Ouro
Cliente_003    397902.19    Prata
Cliente_025    379913.35    Prata
Cliente_020    368325.00    Prata
Cliente_014    361562.73    Prata

=== DISTRIBUIÇÃO POR SEGMENTO ===
segmento
Bronze    11
Prata     33
Ouro       6
Name: count, dtype: int64

=== DISTRIBUIÇÃO PERCENTUAL ===
segmento
Bronze    22.0%
Prata     66.0%
Ouro      12.0%
Name: proportion, dtype: str
